<a href="https://colab.research.google.com/github/MohamedElsagheer95/llm-engineering/blob/main/whisper_pyannote_speach_recognition_and_diarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#install libraries

In [ ]:
pip install faster-whisper pyannote.audio torch transformers

In [ ]:
from huggingface_hub import login
from google.colab import userdata
from transformers import pipeline


In [ ]:
hf_token=userdata.get("HF_TOKEN")
login(hf_token,add_to_git_credential=True)

#Converting audio to mono channel and  passing it through low pass filter

In [ ]:
!ffmpeg -i /content/bakar1.mp3 \
-ac 1 -ar 16000 \
-af "highpass=f=120,lowpass=f=3800,afftdn" \
clean.wav

#Whisper transcription

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(

    "large-v3",
    device="cuda",
    compute_type="float16"
)

segments, info = model.transcribe(
    "/content/clean.wav",
    language="ar",
    beam_size=10,
    best_of=10,
    temperature=0.0,
    vad_filter=True,
    word_timestamps=False
)

segments = list(segments)

In [ ]:
for segment in segments:
  print(segment.text)

 خلاص يا أولاد كل واحد هيختار منظر طبيعي يرسمه بس بشرط
 إن اللوحة تكون بألوان مية
 والحصة الجاية إن شاء الله هستلم منكم اللوحات
 هو لازم ألوان الرسمة بألوان مية؟ ما ينفعش خشب؟
 لا يا حامد ألوان مية لا خشب ولا صفيح
 أنا عايزكوا تتعلموا مهارات جديدة
 واللوحة المرسوبة بألوان المية دي ننشرها على الحبل بعد ما نرسمها عشان تنشف
 لا طبعا يا حسونة أنا هبقى أعلمك تعمل إيه؟
 طب ما بدل ما تعلمني وتتعب نفسك وتتعبني معاك
 ما ينفعش ترسمها لي يا بكر؟
 لا يا حسونة لازم كل واحد يعتمد على نفسه
 لأن اللوحة دي عليها درجات
 يعني أكنها امتحان بزيارة
 بالضبط
 أي دي أنا عادي على مكتبة عم سلم واحنا مروحين
 عشان أجيب علبة ألوان مية
 لأن لو المصروف فضل معايا هيضيع كله على الأكل
 أنا عارف نفسي كويس
 لا تربي الحاجة فضيلة مش بيقولوا كده برضو؟
 هو أنت كنت ساكن فين الأول يا حامد؟
 قبل ما تتنجي لمجرستي
 كنت ساكن في لوكسور
 بس أبويا باعي البيت اللي كنت فيه
 ورجعنا هنا لبلدنا الأصلية
 طيب أبوك باعي بيتكو ليه؟
 عشان شغله في السياحة واجهت بقاله فترة
 وهو دلوقتي بيدور على شغله تاني
 قال له صحيح يا بكار هو أنت بتعرف ترسم؟
 أ

# save transcription in srt file

In [ ]:
def time_formating(sec):
  hr=int(sec//3600)
  mins=int((sec%3600)//60)
  secs=int(sec%60)
  millisec=int((sec-int(sec))*1000)
  return f"{hr:02}:{mins:02}:{secs:02},{millisec:03}"

In [ ]:
with open("audio.srt","w",encoding="utf-8") as f:
  for i , segment in enumerate(segments,start=1):
    start=time_formating(segment.start)
    end=time_formating(segment.end)
    f.write(f"{i}\n")
    f.write(f"{start} --> {end}\n")
    f.write(f"{segment.text.strip()}\n\n")

# Diarization (who speak and when)

In [ ]:
import torch
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1"
)

pipeline.to(torch.device("cuda"))

diarization = pipeline(
    "/content/clean.wav",
    min_speakers=2,
    max_speakers=8,batch_size=32
)

annotation =diarization.speaker_diarization

In [ ]:
annotation = diarization.speaker_diarization

# Now you can iterate over it as expected
for turn, _, speaker in annotation.itertracks(yield_label=True):
    print(f"{speaker}: {turn.start:.2f} → {turn.end:.2f}")

SPEAKER_06: 15.61 → 16.50
SPEAKER_06: 16.89 → 20.15
SPEAKER_06: 20.64 → 26.56
SPEAKER_05: 27.18 → 30.59
SPEAKER_06: 30.88 → 31.98
SPEAKER_06: 32.16 → 35.52
SPEAKER_06: 35.64 → 37.98
SPEAKER_04: 38.88 → 50.15
SPEAKER_04: 50.22 → 57.96
SPEAKER_04: 58.47 → 60.04
SPEAKER_05: 60.04 → 68.41
SPEAKER_05: 75.14 → 86.36
SPEAKER_04: 87.54 → 92.15
SPEAKER_05: 93.70 → 98.50
SPEAKER_05: 98.87 → 104.96
SPEAKER_05: 105.85 → 107.95
SPEAKER_05: 108.55 → 114.04
SPEAKER_05: 114.44 → 117.58
SPEAKER_05: 117.80 → 119.94
SPEAKER_05: 120.94 → 122.56
SPEAKER_05: 122.98 → 124.69
SPEAKER_05: 125.21 → 133.55
SPEAKER_02: 138.96 → 140.95
SPEAKER_02: 140.97 → 144.03
SPEAKER_02: 147.45 → 148.87
SPEAKER_07: 149.36 → 151.26
SPEAKER_04: 151.72 → 153.83
SPEAKER_07: 154.34 → 157.88
SPEAKER_07: 158.03 → 158.07
SPEAKER_04: 158.07 → 158.10
SPEAKER_07: 158.10 → 158.13
SPEAKER_04: 158.13 → 159.03
SPEAKER_07: 159.03 → 160.11
SPEAKER_07: 160.46 → 165.36
SPEAKER_05: 165.68 → 166.33
SPEAKER_05: 166.79 → 167.82
SPEAKER_05: 168.44 → 

# mixing transcription with diarization

In [ ]:
def get_speaker(start, end, annotation):
    best_speaker = "UNKNOWN"
    best_score = 0

    for turn, _, speaker in annotation.itertracks(yield_label=True):
        overlap = max(0, min(end, turn.end) - max(start, turn.start))
        score = overlap / (end - start + 1e-6)

        if score > best_score:
            best_score = score
            best_speaker = speaker

    return best_speaker

In [ ]:
output = []

for seg in segments:
    speaker = get_speaker(seg.start, seg.end, annotation)

    output.append({
        "speaker": speaker,
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip()
    })

    print(f"({seg.start} --> {seg.end}) {speaker}: {seg.text}")

(0.53 --> 20.22) SPEAKER_06:  خلاص يا أولاد كل واحد هيختار منظر طبيعي يرسمه بس بشرط
(20.22 --> 22.84) SPEAKER_06:  إن اللوحة تكون بألوان مية
(22.84 --> 26.66) SPEAKER_06:  والحصة الجاية إن شاء الله هستلم منكم اللوحات
(26.66 --> 30.88) SPEAKER_05:  هو لازم ألوان الرسمة بألوان مية؟ ما ينفعش خشب؟
(30.88 --> 35.24) SPEAKER_06:  لا يا حامد ألوان مية لا خشب ولا صفيح
(35.24 --> 37.9) SPEAKER_06:  أنا عايزكوا تتعلموا مهارات جديدة
(37.9 --> 45.56) SPEAKER_04:  واللوحة المرسوبة بألوان المية دي ننشرها على الحبل بعد ما نرسمها عشان تنشف
(45.56 --> 50.46) SPEAKER_04:  لا طبعا يا حسونة أنا هبقى أعلمك تعمل إيه؟
(50.46 --> 55.34) SPEAKER_04:  طب ما بدل ما تعلمني وتتعب نفسك وتتعبني معاك
(55.34 --> 58.46) SPEAKER_04:  ما ينفعش ترسمها لي يا بكر؟
(58.46 --> 62.8) SPEAKER_05:  لا يا حسونة لازم كل واحد يعتمد على نفسه
(62.8 --> 65.34) SPEAKER_05:  لأن اللوحة دي عليها درجات
(65.34 --> 67.8) SPEAKER_05:  يعني أكنها امتحان بزيارة
(67.9 --> 74.99) SPEAKER_05:  بالضبط
(74.99 --> 77.99) SPEAKER_05:  أي دي أنا عادي 